In [1]:
import numpy as np
import pandas as pd
from keras.layers import Dense, LSTM,GRU, Conv1D, Flatten, MaxPooling1D, Dropout
from keras.models import Sequential
from keras.models import Sequential, Model
from keras.layers import Input, Dense, Dropout, GRU, Conv1D, MaxPooling1D, Flatten, concatenate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score 
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
from IPython.core.pylabtools import figsize

In [2]:
data = pd.read_excel("sm.xls")

In [3]:
train_size = int(len(data)*0.7)
train_dataset, test_dataset = data.iloc[:train_size], data.iloc[train_size:]

In [4]:
# Split train data to X and y
X_train = train_dataset.drop('Eto', axis = 1)
y_train = train_dataset.loc[:,['Eto']]

# Split test data to X and y
X_test = test_dataset.drop('Eto', axis = 1)
y_test = test_dataset.loc[:,['Eto']]

In [5]:
scaler = MinMaxScaler()
X_train= scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
X_train_series = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_series = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

In [7]:
# Build the GRU model
model = Sequential()
model.add(GRU(32, return_sequences=True, input_shape=(5, 1)))
model.add(GRU(8))
model.add(Dense(1))
model.compile(loss='mean_squared_error', optimizer='adam')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru (GRU)                   (None, 5, 32)             3360      
                                                                 
 gru_1 (GRU)                 (None, 8)                 1008      
                                                                 
 dense (Dense)               (None, 1)                 9         
                                                                 
Total params: 4,377
Trainable params: 4,377
Non-trainable params: 0
_________________________________________________________________


In [8]:
model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=32,epochs=30, verbose=1)

Epoch 1/30
56/56 [==============================] - 9s 39ms/step - loss: 20.8103 - val_loss: 4.8464
Epoch 2/30
56/56 [==============================] - 1s 13ms/step - loss: 5.0166 - val_loss: 2.4646
Epoch 3/30
56/56 [==============================] - 1s 12ms/step - loss: 4.0231 - val_loss: 2.1933
Epoch 4/30
56/56 [==============================] - 1s 13ms/step - loss: 3.7226 - val_loss: 2.1624
Epoch 5/30
56/56 [==============================] - 1s 12ms/step - loss: 3.6272 - val_loss: 2.1881
Epoch 6/30
56/56 [==============================] - 1s 12ms/step - loss: 3.6023 - val_loss: 2.2144
Epoch 7/30
56/56 [==============================] - 1s 13ms/step - loss: 3.5946 - val_loss: 2.2299
Epoch 8/30
56/56 [==============================] - 1s 13ms/step - loss: 3.5921 - val_loss: 2.2354
Epoch 9/30
56/56 [==============================] - 1s 13ms/step - loss: 3.5906 - val_loss: 2.2365
Epoch 10/30
56/56 [==============================] - 1s 13ms/step - loss: 3.5901 - val_loss: 2.2451
Epoch 11

In [9]:
ypredtrain = model.predict(X_train)
print(model.evaluate(X_train, y_train))
print("MSE: %.4f" % mean_squared_error(y_train, ypredtrain))

56/56 [==============================] - 2s 4ms/step - loss: 0.1414
0.14144493639469147
MSE: 0.1414


In [10]:
# Evaluation metrices RMSE,MSE,  MAE and R square
print("Train data RMSE: ", math.sqrt(mean_squared_error(y_train,ypredtrain)))
print("Train data MSE: ", mean_squared_error(y_train,ypredtrain))
print("Train data MAE: ", mean_absolute_error(y_train,ypredtrain))
print("Train data R2 score:", r2_score(y_train , ypredtrain ))

Train data RMSE:  0.3760917012317741
Train data MSE:  0.14144496773541004
Train data MAE:  0.2499172397083647
Train data R2 score: 0.9606497092587338


In [11]:
## convert your array into a dataframe
df_pred = pd.DataFrame (ypredtrain)
df_pred.to_excel('Pred_GRU_train__SM.xlsx')

In [13]:
ypredtest = model.predict(X_test)
print(model.evaluate(X_test, y_test))
print("MSE: %.4f" % mean_squared_error(y_test, ypredtest))

24/24 [==============================] - 0s 4ms/step - loss: 0.0634
0.06342105567455292
MSE: 0.0634


In [15]:
# Evaluation metrices RMSE, MSE, MAE and R square
print("Test data RMSE: ", math.sqrt(mean_squared_error(y_test,ypredtest)))
print("Test data MSE: ", mean_squared_error(y_test,ypredtest))
print("Test data MAE: ", mean_absolute_error(y_test,ypredtest))
print("Test data R2 score:", r2_score(y_test , ypredtest ))

Test data RMSE:  0.2518353826603171
Test data MSE:  0.06342105995966833
Test data MAE:  0.1857251271357139
Test data R2 score: 0.9706606500268947


In [16]:
## convert your array into a dataframe
df_pred = pd.DataFrame (ypredtest)
df_pred.to_excel('Pred_GRU_test__SM.xlsx')